In [ ]:
import os
from pathlib import Path
import numpy as np
from tqdm import tqdm

import plotly.graph_objects as go
from Bio.PDB import PDBParser
import py3Dmol

ROOT = Path(os.getcwd()).parents[0]

In [43]:
pdb_file = os.path.join(ROOT, "data", "MDM2_Breaker", "1YCR.pdb")

In [44]:
parser = PDBParser(QUIET=True)

structure = parser.get_structure("MDM2", pdb_file)
structure

<Structure id=MDM2>

In [ ]:
coords = []

for model in structure:
    for chain in model:
        if chain.id == "A":  # Isolate MDM2 (Chain A)
            for residue in tqdm(chain):
                # Filter for 'CA' (Alpha Carbon) - The Backbone "Bead"
                if "CA" in residue:
                    atom = residue["CA"]
                    x, y, z = atom.get_coord()
                    coords.append([x, y, z])
                else:
                    print([x for x in residue])

100%|██████████| 85/85 [00:00<00:00, 134737.66it/s]


In [46]:
[x for x in residue]

[<Atom N>, <Atom CA>, <Atom C>, <Atom O>, <Atom CB>, <Atom CG1>, <Atom CG2>]

In [47]:
np.array(coords).shape

(85, 3)

In [ ]:
def view_protein(pdb_file, highlight_chain="A"):
    view = py3Dmol.view(query=pdb_file)

    # Show the whole protein as a "Cartoon" (Ribbon)
    view.setStyle({"cartoon": {"color": "spectrum"}})

    # Show the Alpha Carbons as spheres
    view.addStyle(
        {"chain": highlight_chain, "atom": "CA"},
        {"sphere": {"radius": 0.5, "color": "red"}},
    )

    view.zoomTo()
    view.show()

view_protein(pdb_file)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [ ]:
def plot_beads(coords, title="MDM2 Alpha Carbons"):
    x, y, z = coords.T

    fig = go.Figure(
        data=[
            go.Scatter3d(
                x=x,
                y=y,
                z=z,
                mode="markers+lines",  # Lines connect the sequence backbone
                marker=dict(size=5, color=z, colorscale="Viridis", opacity=0.8),
                line=dict(color="darkblue", width=2),
            )
        ]
    )

    fig.update_layout(
        title=title,
        scene=dict(xaxis_title="X (Å)", yaxis_title="Y (Å)", zaxis_title="Z (Å)"),
        margin=dict(l=0, r=0, b=0, t=0),
    )

    fig.show()


plot_beads(np.array(coords))